# **MVP: *Análise Exploratória e Boas Práticas***
**Autor:** Gustavo Alves  

**Data:** 2026-03-21

**Matrícula:** 4052025001911

**Dataset:** [Chess Game Dataset](https://www.kaggle.com/datasets/adityajha1504/chesscom-user-games-60000-games)

---


## **1. Escopo, objetivo e definição do problema**

---

### **Definição sobre o esporte xadrez**

- Cada partida ocorre entre dois jogadores, um com as peças `BRANCAS` e outro com as peças `PRETAS`.

- Neste trabalho, chamarei de `BRANCAS` o jogador com peças brancas e de `PRETAS` o jogador com peças pretas.

- A classificação Elo é uma medida de força dos jogadores, atualizada conforme o resultado das partidas, e será uma das principais referências analíticas deste estudo.

&nbsp;

### **Objetivo principal**

- O objetivo é analisar fatores pré-jogo associados à vitória das brancas, estruturando os dados de forma consistente e preparada para análises futuras.

&nbsp;

### **Contexto do problema**

- Dataset publicado no Kaggle: [Chess Game Dataset](https://www.kaggle.com/datasets/adityajha1504/chesscom-user-games-60000-games)

- Conjunto de ~60 mil partidas públicas da Chess.com, coletadas via API (dataset em EN-US; textos do notebook em PT-BR).

- Usarei **somente as informações pré-jogo** (ratings, diferença de rating, controle de tempo, abertura/ECO, etc.). Não avaliarei lances;

&nbsp;

### **Tipo de tarefa**

- Classificação binária: {win, not_win}. O rótulo not_win agrega draw e lose. A mudança do escopo (de multiclasse para binária) foi uma decisão da EDA, pela baixa frequência de draw e pela proximidade lógica entre draw e lose.

&nbsp;

### **Área de aplicação**

- Dados tabulares estruturados com variáveis numéricas e categóricas;

- A análise é conduzida com foco em exploração de dados e preparação para modelagem preditiva.

&nbsp;

### **Valor para o usuário/negócio**

- Jogadores: entender quais fatores pré-jogo mais influenciam o resultado por ritmo de tempo;

- Plataformas/torneios: estimar probabilidades de resultado para pareamentos e alocação de mesas por tempo;

## **2. Reprodutibilidade e ambiente**

---

In [ ]:
# === Imports principais ===
import sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display  # exibir DataFrames no meio da célula
from google.colab import drive       # carregar dados do Google Drive para testes

# === Melhorar visualização no pandas ===
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)

print("Python:", sys.version.split()[0])

# ====== Iniciar Temporizador ======
t0 = time.perf_counter()

Python: 3.12.13


## **3. Dados: carga, entendimento e qualidade**

---

### **3.1 Carga e apresentação do dataset**

In [ ]:
# ====== Carga dos dados ======

# Para CSV no google drive
# drive.mount('/content/drive')
# dataset_url="/content/drive/MyDrive/Colab Notebooks/Analise mercado imobiliario/club_games_dataset_chess_dot_com.csv"

# Para CSV no repo do GitHub
dataset_url = "https://media.githubusercontent.com/media/gustavo-alves-repo/pucrj-mvp-sprint-1-data-analysis-and-best-practices/refs/heads/main/club_games_dataset_chess_dot_com.csv"

# Tipagem e parsing
dtype_map = {
    "white_username": "string",
    "black_username": "string",
    "white_id": "string",
    "black_id": "string",
    "white_rating": "Int64",
    "black_rating": "Int64",
    "white_result": "string",
    "black_result": "string",
    "time_class": "string",
    "time_control": "string",
    "rules": "string",
    "rated": "boolean",
    "fen": "string",
    "pgn": "string",
}

dataset = pd.read_csv(
    dataset_url,
    delimiter=',',
    dtype=dtype_map,
    na_values=["", "NA", "NaN", None],
    low_memory=False,
)

# ====== Visão geral ======
print("Shape:", dataset.shape)

print(
    f"\nMemória aproximada: "
    f"{round(dataset.memory_usage(deep=True).sum() / 1024**2,2)} MB"
)

print("\nPrimeiras linhas (exemplo):")
display(dataset.head().style.set_properties(**{"white-space": "nowrap"}))

Shape: (66879, 14)

Memória aproximada: 190.2 MB

Primeiras linhas (exemplo):


,white_username,black_username,white_id,black_id,white_rating,black_rating,white_result,black_result,time_class,time_control,rules,rated,fen,pgn
0,-Amos-,miniman2804,https://api.chess.com/pub/player/-amos-,https://api.chess.com/pub/player/miniman2804,1708,1608,win,checkmated,daily,1/259200,chess,True,r2r4/p2p1p1p/b6R/n1p1kp2/2P2P2/3BP3/PP5P/4K2R b K f3 1 22,"[Event ""Enjoyable games 2 - Round 1""] [Site ""Chess.com""] [Date ""2013.01.30""] [Round ""-""] [White ""-Amos-""] [Black ""miniman2804""] [Result ""1-0""] [Tournament ""https://www.chess.com/tournament/enjoyable-games-2""] [CurrentPosition ""r2r4/p2p1p1p/b6R/n1p1kp2/2P2P2/3BP3/PP5P/4K2R b K f3 1 22""] [Timezone ""UTC""] [ECO ""E22""] [ECOUrl ""https://www.chess.com/openings/Nimzo-Indian-Defense-Spielmann-Variation""] [UTCDate ""2013.01.30""] [UTCTime ""16:35:14""] [WhiteElo ""1708""] [BlackElo ""1608""] [TimeControl ""1/259200""] [Termination ""-Amos- won by checkmate""] [StartTime ""16:35:14""] [EndDate ""2013.02.01""] [EndTime ""18:14:48""] [Link ""https://www.chess.com/game/daily/64629816""] 1. d4 Nf6 2. c4 e6 3. Nc3 Bb4 4. Qb3 Bxc3+ 5. Qxc3 O-O 6. Bg5 c5 7. dxc5 Nc6 8. Nf3 Qa5 9. Bxf6 gxf6 10. Qxa5 Nxa5 11. e3 Rd8 12. Rd1 Kg7 13. Be2 b6 14. Rd4 bxc5 15. Rg4+ Kh6 16. Bd3 f5 17. Rh4+ Kg6 18. g4 Ba6 19. gxf5+ exf5 20. Ne5+ Kf6 21. Rh6+ Kxe5 22. f4# 1-0"
1,-Amos-,koltcho69,https://api.chess.com/pub/player/-amos-,https://api.chess.com/pub/player/koltcho69,1726,1577,win,resigned,daily,1/172800,chess,True,8/5Q1k/4n1pp/8/7P/2N2b2/PP3P2/5K2 b - - 1 33,"[Event ""Rapid Rats - Board 5""] [Site ""Chess.com""] [Date ""2013.01.19""] [Round ""-""] [White ""-Amos-""] [Black ""koltcho69""] [Result ""1-0""] [Match ""https://www.chess.com/club/matches/219602""] [CurrentPosition ""8/5Q1k/4n1pp/8/7P/2N2b2/PP3P2/5K2 b - - 1 33""] [Timezone ""UTC""] [ECO ""C53""] [ECOUrl ""https://www.chess.com/openings/Giuoco-Piano-Game-Main-Line""] [UTCDate ""2013.01.19""] [UTCTime ""14:29:25""] [WhiteElo ""1726""] [BlackElo ""1577""] [TimeControl ""1/172800""] [Termination ""-Amos- won by resignation""] [StartTime ""14:29:25""] [EndDate ""2013.02.01""] [EndTime ""18:22:03""] [Link ""https://www.chess.com/game/daily/64070770""] 1. e4 e5 2. Nf3 Nc6 3. Bc4 Bc5 4. c3 a6 5. d4 exd4 6. cxd4 Be7 7. Qb3 Na5 8. Qc2 Nxc4 9. Qxc4 d6 10. Nc3 c6 11. O-O h6 12. Re1 Nf6 13. d5 c5 14. e5 dxe5 15. Nxe5 O-O 16. Ng6 Re8 17. Rxe7 Rxe7 18. Nxe7+ Qxe7 19. Bf4 b5 20. d6 Qd7 21. Qxc5 Bb7 22. Qc7 Qxc7 23. dxc7 Nd5 24. Rd1 Nxf4 25. Rd8+ Kh7 26. Rxa8 Bxa8 27. c8=Q Bxg2 28. Qxa6 Bf3 29. Qxb5 Nh3+ 30. Kf1 g6 31. Qd7 Ng5 32. h4 Ne6 33. Qxf7+ 1-0"
2,-Amos-,enhmandah,https://api.chess.com/pub/player/-amos-,https://api.chess.com/pub/player/enhmandah,1727,842,win,resigned,daily,1/172800,chess,True,rn1q1b1r/kb2p1pp/2p5/p1Q5/N1BP2n1/4PN2/1P3PPP/R1B1K2R b KQ - 5 15,"[Event ""CHESS BOARD CLASH - Round 1""] [Site ""Chess.com""] [Date ""2013.02.01""] [Round ""-""] [White ""-Amos-""] [Black ""enhmandah""] [Result ""1-0""] [Tournament ""https://www.chess.com/tournament/just-another-clash""] [CurrentPosition ""rn1q1b1r/kb2p1pp/2p5/p1Q5/N1BP2n1/4PN2/1P3PPP/R1B1K2R b KQ - 5 15""] [Timezone ""UTC""] [ECO ""D00""] [ECOUrl ""https://www.chess.com/openings/Queens-Pawn-Opening-1...d5-2.e3""] [UTCDate ""2013.02.01""] [UTCTime ""11:24:19""] [WhiteElo ""1727""] [BlackElo ""842""] [TimeControl ""1/172800""] [Termination ""-Amos- won by resignation""] [StartTime ""11:24:19""] [EndDate ""2013.02.02""] [EndTime ""17:58:11""] [Link ""https://www.chess.com/game/daily/64714474""] 1. d4 d5 2. e3 c6 3. c4 dxc4 4. Bxc4 b5 5. Bb3 a5 6. Qf3 Bb7 7. Bxf7+ Kd7 8. Qf5+ Kc7 9. Nf3 Nh6 10. Qe5+ Kb6 11. a4 bxa4 12. Nc3 Ng4 13. Nxa4+ Ka6 14. Bc4+ Ka7 15. Qc5+ 1-0"
3,enhmandah,-Amos-,https://api.chess.com/pub/player/enhmandah,https://api.chess.com/pub/player/-amos-,819,1727,checkmated,win,daily,1/172800,chess,True,r3kb1r/pp3ppp/3p1n2/2pKp3/P3P3/1P6/4qP1P/QNB5 w kq - 3 17,"[Event ""CHESS BOARD CLASH - Round 1""] [Site ""Chess.com""] [Date ""2013.02.01""] [Round ""-""] [White ""

### **3.2 Descrição das variáveis**

#### **Nomes de colunas e suas definições**

- **white_username** — nome de usuário das `BRANCAS`
- **black_username** — nome de usuário das `PRETAS`
- **white_id** — link do perfil do jogador das `BRANCAS`
- **black_id** — link do perfil do jogador das `PRETAS`
- **white_rating** — Elo das `BRANCAS` antes da partida
- **black_rating** — Elo das `PRETAS` antes da partida
- **white_result** — resultado do ponto de vista das `BRANCAS` (ver dicionário abaixo)
- **black_result** — resultado do ponto de vista das `PRETAS` (ver dicionário abaixo)
- **time_class** — classe de tempo (bullet, blitz, rapid ou daily)
- **time_control** — controle de tempo no formato base+incremento (ex.: 180+2 = 180s por lado, incremento de 2s por lance).
- **rules** — variante (ex.: chess para xadrez clássico; podem existir outras variantes como chess960)
- **rated** — partida ranqueada (1/True) ou casual (0/False).
- **fen** — Forsyth-Edwards Notation, descreve uma posição do tabuleiro
- **pgn** — Portable Game Notation, texto com metadados e lances da partida.

&nbsp;

#### **Definições de Valores em colunas específicas**

##### **Resultados das Partidas (`white_result` & `black_result`) **

Os rótulos são registrados do ponto de vista de cada cor:

- **win** — vitória (usado em algumas tabelas como rótulo genérico sem detalhar motivo).
- **checkmated** — derrota por xeque-mate.
- **resigned** — derrota por abandono (o jogador desistiu).
- **timeout** — derrota por tempo (relógio zerou).
- **stalemate** — afogamento (sem lance legal e sem xeque) &#8594; empate.
- **agreed** — empate por acordo.
- **repetition** — empate por repetição (três repetições).
- **50move** — empate pela regra dos 50 lances (sem captura nem movimento de peão).
- **insufficient** — material insuficiente para mate &#8594; empate.
- **timevsinsufficient** — um lado cai por tempo, mas o outro não tem material para mate &#8594; empate.
- **abandoned** — partida abandonada/conexão perdida; costuma contar como derrota de quem abandonou.
- **threecheck**, kingofthehill — rótulos associados a variantes (não ao xadrez clássico).

&nbsp;

##### **Controle de Tempo** — time_class

- **bullet** — partidas muito rápidas (geralmente < 3 min por lado).
- **blitz** — partidas rápidas (de 3–10 min por lado).
- **rapid** — partidas rápidas/longas curtas (de 10–30 min por lado).
- **daily** — partidas por correspondência (tempo em dias por lance, ex.: 1/259200 = 3 dias).


Para referências do Chess.com sobre controles de tempo: [Time Controls](https://www.chess.com/terms/chess-time-controls)

### **3.3 Qualidade dos dados**

In [ ]:
print("\nInfo das colunas:")
dataset.info()


Info das colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66879 entries, 0 to 66878
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   white_username  66879 non-null  string 
 1   black_username  66879 non-null  string 
 2   white_id        66879 non-null  string 
 3   black_id        66879 non-null  string 
 4   white_rating    66879 non-null  Int64  
 5   black_rating    66879 non-null  Int64  
 6   white_result    66879 non-null  string 
 7   black_result    66879 non-null  string 
 8   time_class      66879 non-null  string 
 9   time_control    66879 non-null  string 
 10  rules           66879 non-null  string 
 11  rated           66879 non-null  boolean
 12  fen             66879 non-null  string 
 13  pgn             66879 non-null  string 
dtypes: Int64(2), boolean(1), string(11)
memory usage: 6.9 MB


#### **Conclusão**

- **Tipos**: leitura com tipagem explícita — `white_rating`/`black_rating` = Int64; `rated` = boolean; demais = string

- **Nulos**: **0** valores nulos em todas as colunas (ver `info()`)

- **Tamanho & memória**: 66.879 linhas, 14 colunas, ~6.9 MB

- **Próximo passo**: seguir para a EDA (distribuições, criação de `rating_diff`, filtros)

### **3.4 Estatísticas descritivas**

In [ ]:
print("\nEstatísticas:")
display(dataset.describe())


Estatísticas:


,white_rating,black_rating
count,66879.0,66879.0
mean,1247.585729,1246.98273
std,403.895967,403.552072
min,100.0,100.0
25%,976.0,975.0
50%,1252.0,1251.0
75%,1524.0,1524.0
max,3172.0,3172.0


#### **Interpretação das Estatísticas Descritivas**

- Os ratings das brancas e pretas apresentam médias similares (~1247), indicando equilíbrio geral entre os jogadores.

- O desvio padrão elevado (~403) mostra alta variabilidade, sugerindo partidas entre jogadores com diferentes níveis de habilidade — fator relevante para o resultado das partidas.

- Os valores mínimo (100) e máximo (3172) indicam ampla cobertura de níveis, desde iniciantes até avançados.

- De forma geral, os dados são consistentes e reforçam que a diferença de rating pode ser um fator importante na análise do resultado.

- A variável derivada `rating_diff` será criada e analisada nas etapas seguintes.

### **3.5 Análise exploratória resumida (EDA)**

In [ ]:
print("\nDistribuição de white_result & black_result\n")

white = dataset['white_result']
black = dataset['black_result']
resumo = (
    pd.concat([
        white.value_counts().rename('white_total'),
        (white.value_counts(normalize=True)*100).round(1).rename('white_%'),
        black.value_counts().rename('black_total'),
        (black.value_counts(normalize=True)*100).round(1).rename('black_%')
    ], axis=1)
    .reindex(fill_value=0)
)

resumo.loc['Total'] = [
    resumo['white_total'].sum(),
    resumo['white_%'].sum(),
    resumo['black_total'].sum(),
    resumo['black_%'].sum()
]
display(resumo)


#### **Análise — Distribuição dos resultados**

- As vitórias das `BRANCAS` representam aproximadamente 50% das partidas, enquanto as vitórias das `PRETAS` ficam em torno de 47%. Empates representam apenas ~3% do total.

- Essa distribuição reforça a decisão de agrupar empates e derrotas na classe `not_win`, dada a baixa representatividade dos empates.

- O leve predomínio das brancas é consistente com a literatura de xadrez (vantagem do primeiro lance).


In [ ]:
# ====== Distribuição de time_class (gráfico) ======
tc_order = pd.Categorical(
    dataset['time_class'],
    categories=['bullet','blitz','rapid','daily'],
    ordered=True
)

tc_counts = (
    pd.Series(tc_order)
        .value_counts(sort=False)
        .rename('n')
        .to_frame()
        .assign(pct=lambda d: (d['n']/d['n'].sum()*100).round(1))
)

print("Time class — contagem e %\n")
display(tc_counts)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(tc_counts.index.astype(str), tc_counts['n'], color=['#2196F3','#4CAF50','#FF9800','#9C27B0'])
ax.set_ylabel('Número de partidas')
ax.set_xlabel('Classe de tempo')
ax.set_title('Distribuição de partidas por time_class')
for i, (v, p) in enumerate(zip(tc_counts['n'], tc_counts['pct'])):
    ax.text(i, v + 200, f'{p}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


#### **Análise — Distribuição por classe de tempo**

- A classe `blitz` concentra a maior parte das partidas, seguida por `bullet` e `rapid`. Partidas `daily` são residuais.

- Essa distribuição é esperada para a plataforma Chess.com, onde partidas rápidas são predominantes.

- A diferença de volume entre as classes justifica a inclusão de `time_class` como variável categórica na análise, pois o ritmo de jogo pode influenciar o resultado.


In [ ]:
# ====== Distribuição de rules ======
rules_counts = (
    dataset['rules'].value_counts()
        .rename('n')
        .to_frame()
        .assign(pct=lambda d: (d['n']/d['n'].sum()*100).round(1))
)

print("Rules — contagem e %\n")
display(rules_counts)


#### **Análise — Variantes (rules)**

- Mais de 98% das partidas são de xadrez clássico (`chess`). As demais variantes são residuais e serão descartadas na etapa de pré-processamento.


In [ ]:
# ====== Histogramas de rating e rating_diff ======
bins_rd   = [-np.inf, -200, -50, 50, 200, np.inf]
labels_rd = ['≤ -200', '-200 a -50', '-50 a +50', '+50 a +200', '≥ +200']

rating_diff_dataset = (
    dataset
        .assign(rating_diff=lambda d: d['white_rating'] - d['black_rating'],
                diff_bin=lambda d: pd.cut(d['rating_diff'], bins=bins_rd, labels=labels_rd))
        .dropna(subset=['diff_bin','white_result'])
)

print("Histograma de ratings e rating_diff\n")
_ = (
    rating_diff_dataset[['white_rating','black_rating','rating_diff']]
        .hist(bins=80, figsize=(12,4), layout=(1,3), sharey=False)
)
plt.suptitle('Distribuições: White Rating, Black Rating e Rating Diff (White−Black)', y=1.02)
plt.tight_layout()
plt.show()


#### **Análise — Distribuições de rating**

- Os ratings das brancas e pretas apresentam distribuições semelhantes, com pico próximo a 1200, confirmando que a plataforma pareia jogadores de nível similar.

- A variável `rating_diff` (white − black) é centrada em 0, com caudas simétricas e pouca massa além de ±400.

- O desvio padrão elevado (~403) indica alta variabilidade, com partidas entre jogadores de níveis muito diferentes (de 100 a 3172 de Elo).


In [ ]:
# ====== Proporção de resultado por faixa de rating_diff ======
tbl_diff = (
    rating_diff_dataset
        .groupby(['diff_bin','white_result'], observed=True)
        .size()
        .rename('n')
        .reset_index()
)

pct_diff = (
    tbl_diff
        .pivot(index='diff_bin', columns='white_result', values='n')
        .fillna(0)
)
pct_diff = (pct_diff.div(pct_diff.sum(axis=1), axis=0)*100).round(1)

print("Proporção de resultados por faixa de rating_diff (% por linha)\n")
display(pct_diff)

if 'win' in pct_diff.columns:
    fig, ax = plt.subplots(figsize=(6,3))
    ax.bar(pct_diff.index.astype(str), pct_diff['win'])
    ax.set_ylabel('% vitórias das BRANCAS')
    ax.set_xlabel('Faixa de rating_diff (white − black)')
    ax.set_title('Probabilidade de vitória (BRANCAS) por diferença de rating')
    plt.tight_layout()
    plt.show()


#### **Análise — Rating diff vs. resultado**

- O percentual de vitória das brancas cresce monotonicamente com `rating_diff`: de ~15–20% na faixa ≤ -200 até ~80–85% na faixa ≥ +200.

- Na faixa equilibrada (-50 a +50), o win rate é próximo de 50%, com leve vantagem das brancas.

- Esse comportamento confirma que `rating_diff` será a feature mais importante para a análise, pois captura diretamente a diferença de habilidade entre os jogadores.


### **3.6 Distribuição das classes do alvo**


In [ ]:
# ====== Distribuição do target (win vs not_win) ======
win_labels = {"win"}

target_temp = dataset['white_result'].map(
    lambda x: 'win' if x in win_labels else 'not_win'
)

target_counts = target_temp.value_counts()
target_pct = (target_temp.value_counts(normalize=True) * 100).round(1)

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(
    ['win (1)', 'not_win (0)'],
    [target_counts.get('win', 0), target_counts.get('not_win', 0)],
    color=['#4CAF50', '#F44336']
)
for bar, pct in zip(bars, [target_pct.get('win', 0), target_pct.get('not_win', 0)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{pct}%', ha='center', fontsize=10)
ax.set_ylabel('Número de partidas')
ax.set_title('Distribuição do alvo binário (target_bin)')
plt.tight_layout()
plt.show()


#### **Análise — Balanceamento das classes**

- A distribuição do alvo binário é aproximadamente equilibrada: ~50% vitórias das brancas (`win`) contra ~50% não-vitórias (`not_win`).

- Esse balanceamento natural é favorável para uma futura modelagem preditiva, pois não exige técnicas de reamostragem como oversampling ou undersampling.

- O leve predomínio das brancas é consistente com a vantagem estatística conhecida do primeiro lance no xadrez.


### **3.7 Análise combinada: resultado por classe de tempo**


In [ ]:
# ====== Win rate das BRANCAS por time_class ======
win_labels = {"win"}

tc_analysis = (
    dataset
    .assign(is_win=lambda d: d['white_result'].isin(win_labels).astype(int))
    .groupby('time_class')
    .agg(
        total=('is_win', 'size'),
        wins=('is_win', 'sum')
    )
    .assign(win_pct=lambda d: (d['wins'] / d['total'] * 100).round(1))
    .reindex(['bullet', 'blitz', 'rapid', 'daily'])
)

print("Win rate das BRANCAS por time_class\n")
display(tc_analysis)

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(
    tc_analysis.index.astype(str),
    tc_analysis['win_pct'],
    color=['#2196F3','#4CAF50','#FF9800','#9C27B0']
)
ax.set_ylabel('% vitórias das BRANCAS')
ax.set_xlabel('Classe de tempo')
ax.set_title('Win rate das BRANCAS por time_class')
ax.axhline(y=50, color='gray', linestyle='--', linewidth=0.8, label='50%')
for bar, pct in zip(bars, tc_analysis['win_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{pct}%', ha='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()


#### **Análise — Resultado por classe de tempo**

- O win rate das brancas varia entre as classes de tempo, sugerindo que o ritmo de jogo pode influenciar o resultado.

- Em partidas `daily`, a vantagem das brancas tende a ser mais pronunciada, possivelmente porque jogadores com mais tempo conseguem explorar melhor a vantagem posicional do primeiro lance.

- Essa análise reforça a relevância de incluir `time_class` como variável no modelo, pois o comportamento do alvo não é homogêneo entre os ritmos.


### **3.8 Correlação entre variáveis numéricas**


In [ ]:
# ====== Heatmap de correlação (variáveis numéricas pré-jogo) ======
corr_df = (
    dataset
    .assign(
        rating_diff=lambda d: d['white_rating'] - d['black_rating'],
        is_win=lambda d: d['white_result'].isin({"win"}).astype(int),
        rated_int=lambda d: d['rated'].astype(int)
    )
    [['white_rating', 'black_rating', 'rating_diff', 'rated_int', 'is_win']]
)

corr_matrix = corr_df.corr().round(2)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1)

ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr_matrix.columns, fontsize=9)

for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax.text(j, i, f'{corr_matrix.values[i, j]:.2f}',
                ha='center', va='center', fontsize=9,
                color='white' if abs(corr_matrix.values[i, j]) > 0.5 else 'black')

ax.set_title('Correlação entre variáveis numéricas pré-jogo')
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()


#### **Análise — Correlação entre variáveis**

- A variável `rating_diff` apresenta a maior correlação com o alvo (`is_win`), confirmando sua relevância como feature principal.

- `white_rating` e `black_rating` possuem alta correlação entre si, o que é esperado dado o pareamento da plataforma. Essa colinearidade reforça a decisão de usar `rating_diff` como variável derivada em vez dos ratings individuais.

- A variável `rated` apresenta correlação próxima de zero com o resultado, indicando que o fato de a partida ser ranqueada ou casual não influencia significativamente o desfecho.


## **4. Definição do problema, target e variáveis para análise**

---

### **4.1 Descrição do problema**

O objetivo desta análise é entender quais fatores influenciam o resultado de partidas de xadrez, especificamente a vitória das peças brancas.

Trata-se de um problema de aprendizado supervisionado de classificação, no qual se busca prever se as brancas vencem ou não a partida com base em informações disponíveis antes do início do jogo.

&nbsp;

### **4.2 Target**

- Variável alvo conceitual: `white_outcome`
    - `win` → vitória das brancas
    - `not_win` → não vitória das brancas

- Variável alvo binária derivada: `target_bin`
    - `1` → vitória das brancas
    - `0` → não vitória (empate ou derrota)

- **Motivação**: o empate representa uma parcela pequena dos dados (~3%) e, para o objetivo da análise, apresenta comportamento mais próximo da derrota.

&nbsp;

### **4.3 Hipóteses e premissas**

- A diferença de rating entre os jogadores (`rating_diff`) é um dos principais fatores que influenciam o resultado da partida.

- O tipo de controle de tempo (`time_class`) pode impactar o desempenho, especialmente em partidas mais rápidas.

- Partidas ranqueadas (`rated`) podem apresentar comportamentos diferentes de partidas casuais.

&nbsp;

### **4.4 Variáveis utilizadas**

- **Numéricas**:
    - `rating_diff` (diferença entre os ratings)
    - `rated` (indicador binário)

- **Categóricas**:
    - `time_class`, variável categórica relevante para a análise

- Colunas não relevantes para o objetivo foram removidas nesta etapa de preparação.

A transformação de `time_class` em variáveis binárias será realizada na etapa de pré-processamento.

&nbsp;

### **4.5 Premissas e prevenção de vazamento de dados**

- Apenas informações disponíveis antes da partida são consideradas.

- Dados relacionados ao desfecho ou ao andamento do jogo, como `pgn` e `fen`, não são utilizados.

- Apenas partidas de xadrez clássico (`rules == "chess"`) foram mantidas, garantindo maior consistência analítica.


## **5. Pré-processamento dos dados**

---

### **5.1 Filtragem, seleção de colunas e definição do alvo**

In [ ]:
# filtrar somente xadrez clássico (decisão da EDA)
df = dataset.loc[dataset["rules"] == "chess"].copy()

# manter apenas colunas úteis para a análise
df = df[
    [
        "white_rating",
        "black_rating",
        "white_result",
        "time_class",
        "rated",
    ]
].copy()

# normalizar white_result -> white_outcome {win, not_win}
win_labels = {"win"}
draw_labels = {
    "agreed",
    "repetition",
    "stalemate",
    "50move",
    "insufficient",
    "timevsinsufficient",
}
lose_labels = {"resigned", "timeout", "abandoned", "checkmated"}

def white_result_to_outcome(x):
    if x in win_labels:
        return "win"
    if x in draw_labels or x in lose_labels:
        return "not_win"
    return None  # valor inesperado; não deveria ocorrer após os filtros adotados

df["white_outcome"] = df["white_result"].map(white_result_to_outcome)

# conversão de variável booleana para numérica
df["rated"] = df["rated"].astype(int)

# criação de variável derivada
df["rating_diff"] = df["white_rating"] - df["black_rating"]

# criação do alvo binário (1 = win, 0 = not_win)
df["target_bin"] = (df["white_outcome"] == "win").astype(int)

# checagens rápidas
n_before = len(dataset)
n_after = len(df)

print(f"Registros: antes = {n_before} | depois = {n_after} ({(n_after / n_before) * 100:.1f}% mantidos)")
print("white_outcome nulos:", df["white_outcome"].isna().sum())

print("\nDistribuição do alvo (target_bin):")
print(df["target_bin"].value_counts(normalize=True).mul(100).round(1))

print("\nShape do dataset após 5.1:")
print(df.shape)

print("\nPrimeiras linhas:")
display(df.head())

print("\nColunas disponíveis:")
print(df.columns.tolist())

Registros: antes = 66879 | depois = 65778 (98.4% mantidos)
white_outcome nulos: 0

Distribuição do alvo (target_bin):
target_bin
0    50.2
1    49.8
Name: proportion, dtype: float64

Shape do dataset após 5.1:
(65778, 8)

Primeiras linhas:


,white_rating,black_rating,white_result,time_class,rated,white_outcome,rating_diff,target_bin
0,1708,1608,win,daily,1,win,100,1
1,1726,1577,win,daily,1,win,149,1
2,1727,842,win,daily,1,win,885,1
3,819,1727,checkmated,daily,1,not_win,-908,0
4,1729,1116,win,daily,1,win,613,1



Colunas disponíveis:
['white_rating', 'black_rating', 'white_result', 'time_class', 'rated', 'white_outcome', 'rating_diff', 'target_bin']


#### **Conclusão**

- Nesta etapa, foram mantidas apenas partidas da variante clássica (`rules = "chess"`), garantindo maior consistência entre os registros analisados. Em seguida, foram selecionadas apenas as colunas relevantes para o problema, com foco em atributos disponíveis antes da partida.

- Além disso, a variável `white_result` foi consolidada em `white_outcome`, agrupando empates e derrotas na categoria `not_win`. Essa escolha simplifica a análise e reduz a fragmentação das classes observada nos resultados originais.

- Por fim, foram criadas duas variáveis importantes para as etapas seguintes: `rating_diff`, que representa a diferença de rating entre brancas e pretas, e `target_bin`, versão binária do alvo. As checagens realizadas indicam que não houve valores sem mapeamento após a transformação.

### **5.2 Transformação de variáveis categóricas**

A coluna `time_class` é categórica e, portanto, precisa ser convertida para formato numérico para uso em análises posteriores.

Entre as opções possíveis, foram consideradas:

1. `OrdinalEncoder`: atribui um número para cada categoria, mas pode introduzir uma hierarquia artificial entre classes que não possuem ordem natural.

2. `OneHotEncoder`: cria colunas binárias para cada categoria e é uma solução bastante adequada, especialmente em pipelines mais robustos.

3. `pd.get_dummies`: alternativa nativa do pandas, mais simples e suficiente para o volume de dados deste projeto.

Neste trabalho, foi utilizada a função `pd.get_dummies`, por ser uma solução direta, legível e compatível com o porte do dataset.

In [ ]:
# partir do df tratado na etapa anterior
df_model = df[
    [
        "white_rating",
        "black_rating",
        "rated",
        "time_class",
        "rating_diff",
        "white_outcome",
        "target_bin",
    ]
].copy()

# definir categorias esperadas de time_class
tc_categories = ["bullet", "blitz", "rapid", "daily"]

df_model["time_class"] = pd.Categorical(
    df_model["time_class"],
    categories=tc_categories
)

# one-hot encoding com pandas
dummies_tc = pd.get_dummies(df_model["time_class"], prefix="tc", dtype=int)

# garantir ordem consistente das colunas
dummies_tc = dummies_tc.reindex(
    columns=[f"tc_{c}" for c in tc_categories],
    fill_value=0
)

# montar dataset final
df_model = (
    df_model
    .drop(columns=["time_class"])
    .join(dummies_tc)
)

print("Shape final do dataset:")
print(df_model.shape)

print("\nPrimeiras linhas:")
display(df_model.head())

print("\nColunas finais:")
print(df_model.columns.tolist())

Shape final do dataset:
(65778, 10)

Primeiras linhas:


,white_rating,black_rating,rated,rating_diff,white_outcome,target_bin,tc_bullet,tc_blitz,tc_rapid,tc_daily
0,1708,1608,1,100,win,1,0,0,0,1
1,1726,1577,1,149,win,1,0,0,0,1
2,1727,842,1,885,win,1,0,0,0,1
3,819,1727,1,-908,not_win,0,0,0,0,1
4,1729,1116,1,613,win,1,0,0,0,1



Colunas finais:
['white_rating', 'black_rating', 'rated', 'rating_diff', 'white_outcome', 'target_bin', 'tc_bullet', 'tc_blitz', 'tc_rapid', 'tc_daily']


#### **Conclusão**

Nesta etapa, a variável categórica `time_class` foi convertida em variáveis binárias por meio de one-hot encoding. Essa abordagem permite representar cada classe de tempo de forma independente, sem impor relações ordinais artificiais entre elas.

A utilização de `pd.get_dummies` foi considerada suficiente para o volume de dados deste trabalho, oferecendo simplicidade e legibilidade ao processo de pré-processamento. Ao final dessa transformação, obteve-se um dataset final estruturado e pronto para etapas futuras de análise ou modelagem.

### **5.3 Dataset final preparado**

In [ ]:
print("Shape final:", df_model.shape)
display(df_model.head())

Shape final: (65778, 10)


,white_rating,black_rating,rated,rating_diff,white_outcome,target_bin,tc_bullet,tc_blitz,tc_rapid,tc_daily
0,1708,1608,1,100,win,1,0,0,0,1
1,1726,1577,1,149,win,1,0,0,0,1
2,1727,842,1,885,win,1,0,0,0,1
3,819,1727,1,-908,not_win,0,0,0,0,1
4,1729,1116,1,613,win,1,0,0,0,1


Ao final do pré-processamento, o dataset passou a conter apenas variáveis relevantes para o problema, já tratadas e organizadas em formato adequado para análises futuras. As principais transformações incluíram a filtragem das partidas, a definição do alvo binário, a criação da variável `rating_diff` e a codificação da variável categórica `time_class`.

Com isso, a base final encontra-se limpa, consistente e preparada para uso em uma etapa posterior.

## **6. Conclusão**

---

Ao longo deste trabalho, foram realizadas as etapas de compreensão do problema, carga e avaliação da qualidade dos dados, análise exploratória detalhada e pré-processamento. As decisões adotadas em cada etapa foram justificadas e documentadas, seguindo as boas práticas de análise de dados.

Como resultado, obteve-se um conjunto de dados preparado com foco em informações disponíveis antes da partida, incluindo a definição do alvo binário (`target_bin`), a criação da variável derivada `rating_diff` e a codificação da variável categórica `time_class`. A análise exploratória confirmou que `rating_diff` é o fator mais relevante para o resultado, enquanto `time_class` apresenta variações interessantes no win rate entre os ritmos de jogo.

A base encontra-se limpa, consistente e pronta para eventuais etapas futuras de modelagem preditiva, fora do escopo deste trabalho.
